# Raw Ingestion — Food Inspections (dev)

Lands `food_inspections_dev.csv` from the `raw` volume into the `bronze` volume as Parquet, **as-is**: every column is read as STRING (no type conversion, no cleaning, no filtering). Files are organized as `bronze/food_inspections/<ingest-date>/`.

For audit purposes, two columns are appended at write time: `_source` (name of the source file / API) and `_ingested_at` (UTC timestamp recording when the file was received).

In [0]:
# Source and destination paths (bronze layer: raw, untyped landing zone)
from datetime import date, datetime, timezone

SOURCE_CSV = "/Volumes/workspace/foodguard_ai/raw/food_inspections_dev.csv"
BRONZE_BASE = "/Volumes/workspace/foodguard_ai/bronze/food_inspections"
INGEST_DATE = date.today().isoformat()
TARGET_PATH = f"{BRONZE_BASE}/{INGEST_DATE}"

# Audit metadata: which file this came from, and when it was received (UTC).
SOURCE_FILE = SOURCE_CSV.rsplit("/", 1)[-1]
INGESTED_AT = datetime.now(timezone.utc).isoformat()

print(f"Source : {SOURCE_CSV}")
print(f"Target : {TARGET_PATH}")
print(f"Audit  : source={SOURCE_FILE} | ingested_at={INGESTED_AT}")

In [0]:
# Raw read: no schema, no inference — every column stays exactly as in the CSV (STRING).
# multiLine=True keeps records intact: some quoted fields (violations, location) contain embedded newlines.
raw_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .csv(SOURCE_CSV)
)

raw_df.printSchema()
print(f"Rows read: {raw_df.count()}")
display(raw_df.limit(5))

In [0]:
# Write as Parquet: original columns as-is, plus audit columns (_source, _ingested_at).
# Single part file per daily folder, overwrite on re-run.
from pyspark.sql import functions as F

bronze_df = (
    raw_df
    .withColumn("_source", F.lit(SOURCE_FILE))
    .withColumn("_ingested_at", F.lit(INGESTED_AT))
)

(
    bronze_df.coalesce(1)
    .write
    .mode("overwrite")
    .parquet(TARGET_PATH)
)

print(f"Ingested to {TARGET_PATH} (source: {SOURCE_FILE}, ingested_at: {INGESTED_AT})")

In [0]:
# Sanity check: re-read the Parquet and confirm row count, all-STRING schema,
# and audit columns (_source / _ingested_at) survived the round-trip.
from pyspark.sql import functions as F

check_df = spark.read.parquet(TARGET_PATH)

src_count = raw_df.count()
out_count = check_df.count()
same_schema = raw_df.schema == check_df.drop("_source", "_ingested_at").schema
src_values = [r["_source"] for r in check_df.select("_source").distinct().collect()]
null_ingest = check_df.filter(F.col("_ingested_at").isNull()).count()
audit_ok = src_values == [SOURCE_FILE] and null_ingest == 0

print(f"Rows  — source: {src_count} | parquet: {out_count} | match: {src_count == out_count}")
print(f"Schema match (all STRING, minus audit cols): {same_schema}")
print(f"Audit — _source: {src_values} | _ingested_at nulls: {null_ingest}")
assert src_count == out_count, "Row count mismatch!"
assert same_schema, "Schema mismatch — types were altered!"
assert audit_ok, "Audit columns missing or incorrect!"
display(check_df.limit(5))